# LED-off baseline reference diagnostic

Use a random/forced/external-trigger run acquired with the LED off and the same PMT, HV, channel, bandwidth, termination, sampling, and voltage scale as the measurement data. Process one voltage at a time: the notebook builds a robust baseline template, diagnoses coherent pickup and residual noise, flags likely dark-pulse contamination, and saves a voltage-named `.npz` artifact.

The pointwise median—not the ordinary mean—is the primary template because occasional asynchronous dark pulses should not bias it.

In [ ]:
from pathlib import Path
import json
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import uniform_filter1d
from scipy.signal import welch

from lab_tools.io import iter_keysight_chunks
from pmt.io import extract_pmt_voltage

plt.rcParams.update({"figure.figsize": (12, 5), "axes.grid": True, "grid.alpha": 0.35})

## Configuration

In [ ]:
# Put the uploaded reference files here, or change this path.
reference_data_dir = Path("PMT_Data/Baseline_Reference")
# Select exactly one PMT voltage per run, for example '*_800V_*.h5'.
reference_glob = "*_800V_*.h5"
channel = "Channel 3"

# Waveforms are cached on disk and processed in bounded-RAM blocks, so None can
# use every event. Set an integer only when you intentionally want a quick subset.
max_files = None
max_events = None
chunk_size = 512  # RAM also scales with this during per-event diagnostics
cache_dtype = np.float32  # use float64 if full loader precision is required
median_block_samples = 64  # RAM scales as events x this value x dtype size
psd_max_events = 2_000
random_seed = 12345

# A real PMT pulse is broad/unipolar compared with high-frequency bipolar pickup.
# Smoothing is used only to flag contaminated reference traces, never to form
# the saved raw-sample noise distribution.
pulse_test_smoothing_ns = 1.0
pulse_test_snr = 8.0
template_iterations = 2
minimum_clean_fraction = 0.5

# Candidate maximum accidental hardware-trigger rates. Recommendations with
# fewer than minimum_expected_exceedances expected in this reference exposure
# are reported as statistically unsupported rather than silently extrapolated.
target_false_trigger_rates_hz = [100, 1_000, 10_000, 100_000]
minimum_expected_exceedances = 10
trigger_safety_margin_mV = 0.5

# Keep separate collections for bandwidth/filter/termination configurations.
# Batch_analysis.ipynb points each dataset at one of these directories.
reference_collection = "dark_counts"
output_root = Path("baseline_reference")
cache_root = output_root / ".cache"  # use project disk, not a possibly RAM-backed /tmp

## Cache raw reference waveforms on disk

An exact median must still access every selected value at a given time sample. This notebook trades RAM for temporary disk space: the source data are read once into NumPy chunks, then memory-mapped in small time-axis blocks. The cache is deleted after the final artifact is saved.

In [ ]:
reference_files = sorted(reference_data_dir.glob(reference_glob))
if max_files is not None:
    reference_files = reference_files[:max_files]
if not reference_files:
    raise FileNotFoundError(
        f"No reference files matched {reference_data_dir / reference_glob}. "
        "Upload the LED-off run or update reference_data_dir/reference_glob."
    )
reference_voltages_V = {extract_pmt_voltage(path) for path in reference_files}
if len(reference_voltages_V) != 1:
    raise ValueError(
        f"Reference files must contain exactly one voltage; found "
        f"{sorted(reference_voltages_V)}. Narrow reference_glob."
    )
reference_voltage_V = reference_voltages_V.pop()
output_dir = output_root / reference_collection
template_output = output_dir / f"{reference_voltage_V:g}V.npz"
print(f"Reference voltage: {reference_voltage_V:g} V")
print(f"Artifact collection: {output_dir}")

# Clean a previous cache when this cell is rerun, then make a cache whose
# lifetime is tied to the kernel even if a later cell raises an exception.
if "cache_context" in globals():
    cache_context.cleanup()
cache_root.mkdir(parents=True, exist_ok=True)
cache_context = tempfile.TemporaryDirectory(prefix="pmt_baseline_", dir=cache_root)
cache_dir = Path(cache_context.name)
cached_chunks = []
time_ns = None
events_loaded = 0
for chunk_number, chunk in enumerate(iter_keysight_chunks(reference_files, channel=channel, chunk_size=chunk_size)):
    # iter_keysight_chunks exposes standard-unit arrays in this project.
    chunk_time_ns = np.asarray(chunk["time_ns"], dtype=float)
    chunk_voltage_mV = np.asarray(chunk["voltage_mV"], dtype=cache_dtype)
    chunk_time_ns = chunk_time_ns - chunk_time_ns[0]
    if time_ns is None:
        time_ns = chunk_time_ns
    elif len(time_ns) != len(chunk_time_ns) or not np.allclose(time_ns, chunk_time_ns):
        raise ValueError(f"Time axis changed in {chunk['filename']}")
    if max_events is not None:
        remaining = max_events - events_loaded
        if remaining <= 0:
            break
        chunk_voltage_mV = chunk_voltage_mV[:remaining]
    if not len(chunk_voltage_mV):
        break
    cache_path = cache_dir / f"waveforms_{chunk_number:06d}.npy"
    np.save(cache_path, chunk_voltage_mV)
    event_slice = slice(events_loaded, events_loaded + len(chunk_voltage_mV))
    cached_chunks.append((cache_path, event_slice))
    events_loaded = event_slice.stop

if not cached_chunks:
    cache_context.cleanup()
    raise RuntimeError("No waveform events were found in the selected reference files.")
n_samples = len(time_ns)
dt_ns = float(np.median(np.diff(time_ns)))
cache_size_GB = sum(path.stat().st_size for path, _ in cached_chunks) / 1e9
median_block_MB = events_loaded * min(median_block_samples, n_samples) * np.dtype(cache_dtype).itemsize / 1e6
print(f"Cached {events_loaded:,} waveforms from {len(reference_files)} files")
print(f"Shape: ({events_loaded:,}, {n_samples:,}); dt={dt_ns:.5g} ns; duration={time_ns[-1]-time_ns[0]:.3g} ns")
print(f"Temporary disk cache: {cache_dir} ({cache_size_GB:.3g} GB, {np.dtype(cache_dtype).name})")
print(f"One median data block: about {median_block_MB:.1f} MB (NumPy may need additional workspace)")

## Iterative median template and pulse-contamination test

In [ ]:
def robust_rms(values, axis=1):
    center = np.median(values, axis=axis, keepdims=True)
    deviation = np.abs(values - center)
    return 1.4826 * np.median(deviation, axis=axis, overwrite_input=True)

def iter_cached_waveforms():
    for path, event_slice in cached_chunks:
        yield np.load(path, mmap_mode="r"), event_slice

def load_cached_events(indices):
    indices = np.asarray(indices, dtype=int)
    selected = np.empty((len(indices), n_samples), dtype=cache_dtype)
    for waveforms, event_slice in iter_cached_waveforms():
        positions = np.flatnonzero((indices >= event_slice.start) & (indices < event_slice.stop))
        if len(positions):
            selected[positions] = waveforms[indices[positions] - event_slice.start]
    return selected

def pointwise_median(event_mask):
    """Exact pointwise median with RAM proportional to events × median_block_samples."""
    selected_count = int(np.count_nonzero(event_mask))
    if selected_count == 0:
        raise ValueError("Cannot form a median from zero selected events.")
    result = np.empty(n_samples, dtype=float)
    for start in range(0, n_samples, median_block_samples):
        stop = min(start + median_block_samples, n_samples)
        values = np.empty((selected_count, stop - start), dtype=cache_dtype)
        cursor = 0
        for waveforms, event_slice in iter_cached_waveforms():
            local_mask = event_mask[event_slice]
            count = int(np.count_nonzero(local_mask))
            if count:
                values[cursor:cursor + count] = waveforms[local_mask, start:stop]
                cursor += count
        result[start:stop] = np.median(values, axis=0, overwrite_input=True)
    return result

def event_statistics(template_mV):
    robust_rms_mV = np.empty(events_loaded, dtype=float)
    pulse_statistic = np.empty(events_loaded, dtype=float)
    event_offset_mV = np.empty(events_loaded, dtype=float)
    template_for_chunks = np.asarray(template_mV, dtype=cache_dtype)
    for waveforms, event_slice in iter_cached_waveforms():
        residual = np.asarray(waveforms) - template_for_chunks
        offsets = np.median(residual, axis=1)
        residual -= offsets[:, None]
        rms = robust_rms(residual)
        smoothed = uniform_filter1d(residual, size=smooth_samples, axis=1, mode="nearest")
        negative_excursion_mV = -np.min(smoothed, axis=1)
        robust_rms_mV[event_slice] = rms
        event_offset_mV[event_slice] = offsets
        pulse_statistic[event_slice] = np.divide(
            negative_excursion_mV, rms,
            out=np.full(len(residual), np.inf), where=rms > 0,
        )
    return robust_rms_mV, pulse_statistic, event_offset_mV

def clean_mean(event_mask):
    total = np.zeros(n_samples, dtype=float)
    count = 0
    for waveforms, event_slice in iter_cached_waveforms():
        local_mask = event_mask[event_slice]
        if np.any(local_mask):
            total += np.sum(waveforms[local_mask], axis=0, dtype=float)
            count += int(np.count_nonzero(local_mask))
    return total / count

def residual_sample_block(event_mask, template_mV, event_offset_mV, start, stop):
    selected_count = int(np.count_nonzero(event_mask))
    residual = np.empty((selected_count, stop - start), dtype=cache_dtype)
    cursor = 0
    for waveforms, event_slice in iter_cached_waveforms():
        local_mask = event_mask[event_slice]
        count = int(np.count_nonzero(local_mask))
        if count:
            residual[cursor:cursor + count] = (
                waveforms[local_mask, start:stop]
                - template_mV[None, start:stop]
                - event_offset_mV[event_slice][local_mask, None]
            )
            cursor += count
    return residual

def residual_summaries(event_mask, template_mV, event_offset_mV):
    sigma = np.empty(n_samples, dtype=float)
    quantiles = np.empty((4, n_samples), dtype=float)
    for start in range(0, n_samples, median_block_samples):
        stop = min(start + median_block_samples, n_samples)
        residual = residual_sample_block(
            event_mask, template_mV, event_offset_mV, start, stop
        )
        np.abs(residual, out=residual)
        sigma[start:stop] = 1.4826 * np.median(
            residual, axis=0, overwrite_input=True
        )
        # Reload this small block because the in-place median destroys it.
        residual = residual_sample_block(
            event_mask, template_mV, event_offset_mV, start, stop
        )
        quantiles[:, start:stop] = np.quantile(
            residual, [0.001, 0.01, 0.99, 0.999],
            axis=0, overwrite_input=True,
        )
    return sigma, quantiles

smooth_samples = max(1, int(round(pulse_test_smoothing_ns / dt_ns)))
clean_mask = np.ones(events_loaded, dtype=bool)
for iteration in range(template_iterations):
    baseline_template_mV = pointwise_median(clean_mask)
    event_robust_rms_mV, pulse_test_statistic, _ = event_statistics(baseline_template_mV)
    clean_mask = pulse_test_statistic < pulse_test_snr
    print(f"Iteration {iteration + 1}: {clean_mask.sum():,}/{len(clean_mask):,} ({clean_mask.mean():.2%}) reference traces retained")

if clean_mask.mean() < minimum_clean_fraction:
    raise RuntimeError(
        f"Only {clean_mask.mean():.1%} of traces passed. Inspect the plots and "
        "adjust pulse_test_smoothing_ns/pulse_test_snr before saving a template."
    )

events_retained = int(clean_mask.sum())
baseline_template_mV = pointwise_median(clean_mask)
event_robust_rms_mV, pulse_test_statistic, event_offset_mV = event_statistics(baseline_template_mV)
mean_waveform_mV = clean_mean(clean_mask)
residual_sigma_mV, residual_quantiles_mV = residual_summaries(
    clean_mask, baseline_template_mV, event_offset_mV
)
residual_q001_mV, residual_q01_mV, residual_q99_mV, residual_q999_mV = residual_quantiles_mV

## Template and contamination diagnostics

In [ ]:
rng = np.random.default_rng(random_seed)
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sample = rng.choice(events_loaded, size=min(30, events_loaded), replace=False)
sample_waveforms_mV = load_cached_events(sample)
axes[0, 0].plot(time_ns, sample_waveforms_mV.T, color="0.6", alpha=0.18, lw=0.7)
axes[0, 0].plot(time_ns, baseline_template_mV, color="tab:blue", lw=2, label="Clean pointwise median")
axes[0, 0].plot(time_ns, mean_waveform_mV, color="tab:orange", lw=1.2, label="Clean mean")
axes[0, 0].set(xlabel="Time [ns]", ylabel="Voltage [mV]", title="Raw references and templates")
axes[0, 0].legend()

axes[0, 1].hist(pulse_test_statistic, bins=150, log=True)
axes[0, 1].axvline(pulse_test_snr, color="tab:red", label=f"Contamination threshold = {pulse_test_snr:g}")
axes[0, 1].set(xlabel="Smoothed negative excursion / robust RMS", ylabel="Events", title="Likely pulse contamination")
axes[0, 1].legend()

axes[1, 0].fill_between(time_ns, residual_q001_mV, residual_q999_mV, alpha=0.22, label="0.1–99.9%")
axes[1, 0].fill_between(time_ns, residual_q01_mV, residual_q99_mV, alpha=0.35, label="1–99%")
axes[1, 0].plot(time_ns, residual_sigma_mV, lw=1, label="Robust sigma")
axes[1, 0].set(xlabel="Time [ns]", ylabel="Residual [mV]", title="Empirical residual-noise envelope")
axes[1, 0].legend()

axes[1, 1].hist(event_robust_rms_mV[clean_mask], bins=100, alpha=0.8, label="Retained references")
axes[1, 1].set(xlabel="Per-event robust RMS [mV]", ylabel="Events", title="Residual noise scale")
axes[1, 1].legend()
fig.tight_layout()

In [ ]:
flagged = np.flatnonzero(~clean_mask)
if len(flagged):
    show = rng.choice(flagged, size=min(20, len(flagged)), replace=False)
    flagged_waveforms_mV = load_cached_events(show)
    fig, ax = plt.subplots(figsize=(15, 6))
    ax.plot(time_ns, (flagged_waveforms_mV - baseline_template_mV).T, alpha=0.35, lw=0.8)
    ax.axhline(0, color="black", lw=0.8)
    ax.set(xlabel="Time [ns]", ylabel="Template-subtracted voltage [mV]", title=f"Likely pulse-contaminated references (showing {len(show)} of {len(flagged)})")
    fig.tight_layout()
else:
    print("No reference traces were flagged as pulse-contaminated.")

## Empirical hardware-trigger recommendation

This calculation uses the raw negative excursion relative to the global baseline center, so coherent pickup remains included just as it would be seen by the oscilloscope trigger. A recommendation is trustworthy only when the reference exposure contains enough windows to measure the requested tail probability.

In [ ]:
window_duration_s = (time_ns[-1] - time_ns[0] + dt_ns) * 1e-9
reference_exposure_s = events_retained * window_duration_s
global_baseline_mV = float(np.median(baseline_template_mV))
hardware_negative_excursion_parts_mV = []
for waveforms, event_slice in iter_cached_waveforms():
    local_mask = clean_mask[event_slice]
    if np.any(local_mask):
        hardware_negative_excursion_parts_mV.append(
            global_baseline_mV - np.min(waveforms[local_mask], axis=1)
        )
hardware_negative_excursion_mV = np.concatenate(hardware_negative_excursion_parts_mV)

trigger_rows = []
for target_rate_hz in target_false_trigger_rates_hz:
    probability_per_window = 1.0 - np.exp(-target_rate_hz * window_duration_s)
    expected_exceedances = events_retained * probability_per_window
    noise_excursion_quantile_mV = float(np.quantile(
        hardware_negative_excursion_mV, 1.0 - probability_per_window
    ))
    recommended_excursion_mV = noise_excursion_quantile_mV + trigger_safety_margin_mV
    trigger_rows.append({
        "target_false_rate_hz": target_rate_hz,
        "probability_per_window": probability_per_window,
        "expected_tail_events": expected_exceedances,
        "noise_excursion_quantile_mV": noise_excursion_quantile_mV,
        "safety_margin_mV": trigger_safety_margin_mV,
        "recommended_negative_excursion_mV": recommended_excursion_mV,
        "recommended_scope_level_mV": global_baseline_mV - recommended_excursion_mV,
        "supported_by_exposure": expected_exceedances >= minimum_expected_exceedances,
    })
trigger_recommendations = pd.DataFrame(trigger_rows)
print(f"Clean reference exposure: {reference_exposure_s:.6g} s in {events_retained:,} windows")
print(f"Global baseline level: {global_baseline_mV:.3f} mV")
display(trigger_recommendations)
if not trigger_recommendations["supported_by_exposure"].all():
    print("Warning: unsupported rows probe a rarer tail than this reference exposure can measure reliably. Collect more random-trigger windows before using them.")

sorted_excursion_mV = np.sort(hardware_negative_excursion_mV)
exceedance_counts = np.arange(len(sorted_excursion_mV), 0, -1)
empirical_false_rate_hz = exceedance_counts / reference_exposure_s
fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(sorted_excursion_mV, empirical_false_rate_hz, label="Empirical noise-trigger rate")
for row in trigger_rows:
    if row["supported_by_exposure"]:
        ax.scatter(row["recommended_negative_excursion_mV"], row["target_false_rate_hz"], s=45)
ax.set(xlabel="Negative trigger excursion below baseline [mV]", ylabel="Estimated accidental rate [Hz]", title="Noise-only trigger threshold from LED-off references")
ax.legend()
fig.tight_layout()

## Frequency-domain diagnostic

In [ ]:
fs_hz = 1e9 / dt_ns
psd_indices = np.flatnonzero(clean_mask)[:psd_max_events]
psd_parts = []
template_for_chunks = np.asarray(baseline_template_mV, dtype=cache_dtype)
for start in range(0, len(psd_indices), chunk_size):
    batch_indices = psd_indices[start:start + chunk_size]
    residual_batch = load_cached_events(batch_indices)
    residual_batch -= template_for_chunks
    residual_batch -= event_offset_mV[batch_indices, None].astype(cache_dtype)
    frequency_hz, psd_batch = welch(
        residual_batch, fs=fs_hz, axis=1, nperseg=min(2048, n_samples)
    )
    psd_parts.append(psd_batch)
psd = np.concatenate(psd_parts, axis=0)
median_psd = np.median(psd, axis=0, overwrite_input=True)
fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(frequency_hz / 1e6, median_psd)
ax.set(xlabel="Frequency [MHz]", ylabel="Median PSD [mV²/Hz]", title="Residual frequency content after median-template subtraction")
fig.tight_layout()

## Save reusable reference artifact

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
metadata = {
    "voltage_V": reference_voltage_V,
    "reference_collection": reference_collection,
    "channel": channel,
    "reference_data_dir": str(reference_data_dir),
    "reference_files": [str(path) for path in reference_files],
    "events_loaded": int(events_loaded),
    "events_retained": events_retained,
    "clean_fraction": float(clean_mask.mean()),
    "dt_ns": dt_ns,
    "cache_dtype": np.dtype(cache_dtype).name,
    "median_block_samples": median_block_samples,
    "pulse_test_smoothing_ns": pulse_test_smoothing_ns,
    "pulse_test_snr": pulse_test_snr,
    "reference_exposure_s": reference_exposure_s,
    "global_baseline_mV": global_baseline_mV,
}
np.savez_compressed(
    template_output,
    time_ns=time_ns,
    baseline_template_mV=baseline_template_mV,
    mean_waveform_mV=mean_waveform_mV,
    residual_sigma_mV=residual_sigma_mV,
    residual_q001_mV=residual_q001_mV,
    residual_q01_mV=residual_q01_mV,
    residual_q99_mV=residual_q99_mV,
    residual_q999_mV=residual_q999_mV,
    hardware_negative_excursion_mV=hardware_negative_excursion_mV,
    trigger_target_false_rate_hz=trigger_recommendations["target_false_rate_hz"].to_numpy(),
    trigger_recommended_scope_level_mV=trigger_recommendations["recommended_scope_level_mV"].to_numpy(),
    trigger_supported_by_exposure=trigger_recommendations["supported_by_exposure"].to_numpy(),
    metadata_json=json.dumps(metadata),
)
print(f"Saved baseline reference to {template_output.resolve()}")
display(metadata)
if cache_dir.exists():
    cache_context.cleanup()
    print(f"Removed temporary waveform cache ({cache_size_GB:.3g} GB).")